# Exploratory Data Analysis (EDA) - Student Performance Prediction

This notebook performs a comprehensive Exploratory Data Analysis on the Student Performance dataset (`student-mat.csv`). The goal is to understand the factors influencing the final grade ($G3$) and identify key relationships to inform our machine learning modeling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

## 1. Load and Inspect the Dataset

We read the raw dataset from `../data/raw/student-mat.csv`. The fields are delimited by semicolons.

In [ ]:
df = pd.read_csv("../data/raw/student-mat.csv", sep=";")
print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns\n")
df.head()

## 2. Missing Values Analysis

Let's check for any missing values in the dataset.

In [ ]:
missing_values = df.isnull().sum()
print("Missing Values per Column:")
print(missing_values[missing_values > 0] if missing_values.sum() > 0 else "No missing values found! Check passed.")

## 3. Statistical Summary

Let's look at the summary statistics for the numerical features.

In [ ]:
df.describe().T

## 4. Distribution of Final Grades ($G3$)

We visualize the distribution of $G3$. We also notice that some students scored 0. Let's see if this creates a bimodal distribution.

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(df['G3'], bins=20, kde=True, color='royalblue')
plt.title('Distribution of Final Grades (G3)')
plt.xlabel('G3 Grade (0-20)')
plt.ylabel('Count')
os.makedirs('../src/app/assets', exist_ok=True)
plt.savefig('../src/app/assets/grade_distribution.png', dpi=300)
plt.show()

## 5. Correlation Analysis

We calculate the correlations between numerical features and G3 to find the strongest linear relationships.

In [ ]:
# Select numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlation Matrix of Numerical Features')
plt.savefig('../src/app/assets/correlation_heatmap.png', dpi=300)
plt.show()

Let's see the correlations specifically with G3.

In [ ]:
corr_with_g3 = corr_matrix['G3'].sort_values(ascending=False)
print("Correlations with G3:")
print(corr_with_g3)

## 6. Feature Relationships and Visualizations

Let's look at key demographic, social, and academic factors and their relationships with $G3$.

### A. Failures vs Final Grade ($G3$)

Past failures is highly negatively correlated with the final grade.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x='failures', y='G3', data=df, palette='Set2')
plt.title('Final Grade (G3) by Number of Past Failures')
plt.xlabel('Number of Past Failures')
plt.ylabel('G3 Grade')
plt.savefig('../src/app/assets/failures_vs_g3.png', dpi=300)
plt.show()

### B. Study Time vs Final Grade ($G3$)

Does studying more lead to higher grades? Let's check.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x='studytime', y='G3', data=df, palette='Set1')
plt.title('Final Grade (G3) by Weekly Study Time')
plt.xlabel('Study Time (1: <2h, 2: 2-5h, 3: 5-10h, 4: >10h)')
plt.ylabel('G3 Grade')
plt.savefig('../src/app/assets/studytime_vs_g3.png', dpi=300)
plt.show()

### C. Alcohol Consumption vs Final Grade ($G3$)

Let's look at weekend alcohol consumption (`Walc`) and workday alcohol consumption (`Dalc`) vs G3.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(ax=axes[0], x='Dalc', y='G3', data=df, palette='Reds')
axes[0].set_title('Workday Alcohol Consumption vs G3')
axes[0].set_xlabel('Workday Alcohol Consumption (1: very low to 5: very high)')
axes[0].set_ylabel('G3 Grade')

sns.boxplot(ax=axes[1], x='Walc', y='G3', data=df, palette='Reds')
axes[1].set_title('Weekend Alcohol Consumption vs G3')
axes[1].set_xlabel('Weekend Alcohol Consumption (1: very low to 5: very high)')
axes[1].set_ylabel('G3 Grade')

plt.savefig('../src/app/assets/alcohol_vs_g3.png', dpi=300)
plt.show()

### D. Absences vs Final Grade ($G3$)

Let's see the scatter plot of absences vs final grade.

In [ ]:
plt.figure(figsize=(9, 5))
sns.scatterplot(x='absences', y='G3', data=df, alpha=0.7, color='crimson')
plt.title('School Absences vs Final Grade (G3)')
plt.xlabel('Number of Absences')
plt.ylabel('G3 Grade')
plt.savefig('../src/app/assets/absences_vs_g3.png', dpi=300)
plt.show()

## 7. Key Findings Summary

Based on the visualizations and statistical checks, we observe:
1. **Past Failures**: The strongest negative correlation with final grades. Students with zero failures perform significantly better.
2. **Study Time**: Students who spend more time studying tend to score higher, although studying >10 hours has diminishing returns compared to 5-10 hours.
3. **G1 and G2**: Extremely strong positive correlations ($r \approx 0.80$ to $0.90$) with final grade G3. They reflect ongoing academic progress.
4. **Alcohol and Social factors**: High workday and weekend alcohol consumption correlates with lower median grades and a wider variance of student performance.
5. **Absences**: Higher absences have a weak negative relationship, with some outliers having high absences but decent grades, and others failing.